# COLM Role Abstraction — Exploratory Analysis

Qualitative and quantitative validation of the role abstraction stage output.
Compares original (character-specific) norms against their abstracted (functional social role) rewrites.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'))

ABSTRACTION_PATH = os.environ.get('ABSTRACTION_PATH',
    '/share/pierson/matt/UAIR/outputs/2026-03-21_historical_norms/18-37-21/role_abstraction_standalone/outputs/role_abstraction/abstracted_norms.parquet')

df = pd.read_parquet(ABSTRACTION_PATH)

# Normalize types
df['norm_quality_passed'] = df['norm_quality_passed'].astype(bool)
df['role_abstraction_failed'] = df['role_abstraction_failed'].astype(str).str.lower().eq('true')

# Working subset: rows that had norms and were processed
processed = df[df['has_norms'] & ~df['role_abstraction_failed']].copy()

print(f'Loaded {len(df):,} rows, {len(processed):,} successfully abstracted')
print(f'Abstraction failures: {df["role_abstraction_failed"].sum():,}')
print(f'Columns: {len(df.columns)}')

## 1. Character name removal — did it work?

Check whether the abstracted norms still contain character names. Uses the same blocklist
and title pattern as the extraction quality validator.

In [ ]:
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))
from dagspaces.historical_norms.stages.norm_extraction import _get_character_blocklist, _TITLE_PATTERN

blocklist = _get_character_blocklist(None)

def has_character_name(text, blocklist=blocklist):
    """Check if text contains any character name from blocklist or titled pattern."""
    if pd.isna(text) or not text:
        return False
    text_lower = text.lower()
    for name in blocklist:
        if re.search(r'\b' + re.escape(name) + r'\b', text_lower):
            return True
    if _TITLE_PATTERN.search(text):
        return True
    return False

# Check each rewritten field
for field, orig_field in [
    ('raz_norm_subject', 'orig_raz_norm_subject'),
    ('raz_norm_act', 'orig_raz_norm_act'),
    ('raz_condition_of_application', 'orig_raz_condition_of_application'),
    ('raz_norm_articulation', 'orig_raz_norm_articulation'),
]:
    orig_has = processed[orig_field].apply(has_character_name).sum()
    new_has = processed[field].apply(has_character_name).sum()
    print(f'{field}:')
    print(f'  Before: {orig_has:,} ({orig_has/len(processed)*100:.1f}%) had character names')
    print(f'  After:  {new_has:,} ({new_has/len(processed)*100:.1f}%) have character names')
    print(f'  Reduction: {(1 - new_has/max(orig_has,1))*100:.1f}%')
    print()

## 2. Residual character names — what slipped through?

In [ ]:
residual = processed[processed['raz_norm_articulation'].apply(has_character_name)]

print(f'Residual norms with character names in articulation: {len(residual):,} / {len(processed):,}')
print()

if len(residual) > 0:
    # Which names are still present?
    name_counts = {}
    for _, row in residual.iterrows():
        art = (row['raz_norm_articulation'] or '').lower()
        for name in blocklist:
            if re.search(r'\b' + re.escape(name) + r'\b', art):
                name_counts[name] = name_counts.get(name, 0) + 1
    
    print('Most common residual names:')
    for name, count in sorted(name_counts.items(), key=lambda x: -x[1])[:15]:
        print(f'  {name}: {count}')
    
    print(f'\nResidual by book:')
    print(residual['book_title'].value_counts().to_string())
    
    print(f'\nSample residual norms:')
    for _, row in residual.sample(min(5, len(residual)), random_state=42).iterrows():
        print(f'  [{row["book_title"]}]')
        print(f'    ORIG: {str(row["orig_raz_norm_articulation"])[:120]}')
        print(f'    NEW:  {str(row["raz_norm_articulation"])[:120]}')
        print()

## 3. Subject rewrite quality — side-by-side comparison

Compare original vs. abstracted subjects for each novel. Shows how character names
were generalized into functional social roles.

In [ ]:
sample_books = [
    'Pride and Prejudice', 'Nineteen Eighty-Four', 'Les Misérables',
    'Anna Karenina', 'The Count of Monte Cristo', 'Bleak House',
    'Middlemarch', 'The Picture of Dorian Gray',
    "Alice's Adventures in Wonderland", 'The Age of Innocence',
]

for book in sample_books:
    subset = processed[processed['book_title'] == book]
    # Pick norms where the subject actually changed
    changed = subset[subset['raz_norm_subject'] != subset['orig_raz_norm_subject']]
    if changed.empty:
        continue
    samples = changed.sample(min(3, len(changed)), random_state=42)
    print(f'{"═" * 80}')
    print(f'{book} ({len(changed):,} subjects changed out of {len(subset):,})')
    print(f'{"═" * 80}')
    for _, row in samples.iterrows():
        print(f'  ORIG subject: {row["orig_raz_norm_subject"]}')
        print(f'  NEW subject:  {row["raz_norm_subject"]}')
        print(f'  Rationale:    {str(row["role_rationale"])[:200]}')
        print()

## 4. Rewrite rate — what fraction of norms were changed vs. passed through?

In [ ]:
processed['subject_changed'] = processed['raz_norm_subject'] != processed['orig_raz_norm_subject']
processed['act_changed'] = processed['raz_norm_act'] != processed['orig_raz_norm_act']
processed['condition_changed'] = processed['raz_condition_of_application'] != processed['orig_raz_condition_of_application']
processed['articulation_changed'] = processed['raz_norm_articulation'] != processed['orig_raz_norm_articulation']
processed['any_changed'] = processed['subject_changed'] | processed['act_changed'] | processed['condition_changed'] | processed['articulation_changed']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall rewrite rates
fields = ['subject_changed', 'act_changed', 'condition_changed', 'articulation_changed', 'any_changed']
rates = [processed[f].mean() * 100 for f in fields]
labels = ['Subject', 'Act', 'Condition', 'Articulation', 'Any field']
axes[0].barh(labels, rates, color='steelblue')
axes[0].set_xlabel('% of norms rewritten')
axes[0].set_title('Rewrite Rate by Field')
for i, v in enumerate(rates):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center')

# Rewrite rate by novel
rewrite_by_book = processed.groupby('book_title')['any_changed'].mean().sort_values() * 100
rewrite_by_book.plot.barh(ax=axes[1], color='darkorange')
axes[1].set_xlabel('% of norms rewritten')
axes[1].set_title('Rewrite Rate by Novel')

plt.tight_layout()
plt.show()

# Was quality_passed correlated with rewrites?
flagged_rewrite = processed[~processed['norm_quality_passed']]['any_changed'].mean() * 100
passed_rewrite = processed[processed['norm_quality_passed']]['any_changed'].mean() * 100
print(f'Rewrite rate for quality-flagged norms:  {flagged_rewrite:.1f}%')
print(f'Rewrite rate for quality-passed norms:   {passed_rewrite:.1f}%')

## 5. Subject length before/after — are roles more descriptive?

In [ ]:
processed['orig_subj_len'] = processed['orig_raz_norm_subject'].str.len()
processed['new_subj_len'] = processed['raz_norm_subject'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram overlay
axes[0].hist(processed['orig_subj_len'], bins=50, alpha=0.6, label='Original', color='#d62728', edgecolor='black')
axes[0].hist(processed['new_subj_len'], bins=50, alpha=0.6, label='Abstracted', color='#2ca02c', edgecolor='black')
axes[0].set_xlabel('Subject length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Subject Length Distribution')
axes[0].legend()

# Per-novel comparison
len_by_book = processed.groupby('book_title').agg(
    orig_mean=('orig_subj_len', 'mean'),
    new_mean=('new_subj_len', 'mean'),
).sort_values('new_mean')
len_by_book.plot.barh(ax=axes[1], color=['#d62728', '#2ca02c'])
axes[1].set_xlabel('Mean subject length (chars)')
axes[1].set_title('Subject Length by Novel')
axes[1].legend(['Original', 'Abstracted'])

plt.tight_layout()
plt.show()

print(f'Original subject length:   mean={processed["orig_subj_len"].mean():.0f}, median={processed["orig_subj_len"].median():.0f}')
print(f'Abstracted subject length: mean={processed["new_subj_len"].mean():.0f}, median={processed["new_subj_len"].median():.0f}')

## 6. Prescriptive element preservation — was deontic force kept intact?

In [ ]:
# Check that prescriptive_element and normative_force were not changed
# (These are passthrough columns — the LLM should not have touched them)
# We can check by looking at whether the articulation still uses the original deontic word

force_dist_orig = processed['raz_normative_force'].value_counts()
print('Normative force distribution (should be unchanged from extraction):')
print(force_dist_orig)
print()

# Spot-check: does the abstracted articulation still use the correct prescriptive element?
mismatches = []
for _, row in processed.sample(min(500, len(processed)), random_state=42).iterrows():
    pe = str(row['raz_prescriptive_element']).lower()
    art = str(row['raz_norm_articulation']).lower()
    # Check if prescriptive element appears in articulation
    if pe and pe not in art:
        mismatches.append({
            'book': row['book_title'],
            'pe': row['raz_prescriptive_element'],
            'articulation': row['raz_norm_articulation'][:120],
        })

print(f'Prescriptive element missing from articulation: {len(mismatches)} / 500 sampled')
if mismatches:
    print('\nSample mismatches:')
    for m in mismatches[:5]:
        print(f'  [{m["book"]}] PE="{m["pe"]}"')
        print(f'    Art: {m["articulation"]}')

## 7. Role rationale quality — sample rationales across novels

In [ ]:
for book in sample_books:
    subset = processed[(processed['book_title'] == book) & processed['any_changed']]
    if subset.empty:
        continue
    row = subset.sample(1, random_state=7).iloc[0]
    print(f'{"═" * 80}')
    print(f'{book}')
    print(f'{"─" * 80}')
    print(f'ORIGINAL:')
    print(f'  Subject: {row["orig_raz_norm_subject"]}')
    print(f'  Act:     {row["orig_raz_norm_act"]}')
    print(f'  Cond:    {row["orig_raz_condition_of_application"]}')
    print(f'  Artic:   {str(row["orig_raz_norm_articulation"])[:200]}')
    print(f'ABSTRACTED:')
    print(f'  Subject: {row["raz_norm_subject"]}')
    print(f'  Act:     {row["raz_norm_act"]}')
    print(f'  Cond:    {row["raz_condition_of_application"]}')
    print(f'  Artic:   {str(row["raz_norm_articulation"])[:200]}')
    print(f'RATIONALE: {row["role_rationale"]}')
    print()

## 8. No-op passthrough — did already-abstract norms survive unchanged?

In [ ]:
# Norms that were already abstract (quality-passed, no character names in original)
already_abstract = processed[
    processed['norm_quality_passed'] & 
    ~processed['orig_raz_norm_subject'].apply(has_character_name)
]

unchanged = already_abstract[~already_abstract['any_changed']]
changed_anyway = already_abstract[already_abstract['any_changed']]

print(f'Already-abstract norms: {len(already_abstract):,}')
print(f'  Kept unchanged:       {len(unchanged):,} ({len(unchanged)/len(already_abstract)*100:.1f}%)')
print(f'  Changed anyway:       {len(changed_anyway):,} ({len(changed_anyway)/len(already_abstract)*100:.1f}%)')
print()

if len(changed_anyway) > 0:
    print('Sample norms changed despite being already abstract:')
    for _, row in changed_anyway.sample(min(5, len(changed_anyway)), random_state=42).iterrows():
        print(f'  [{row["book_title"]}]')
        print(f'    ORIG subject: {row["orig_raz_norm_subject"]}')
        print(f'    NEW subject:  {row["raz_norm_subject"]}')
        print(f'    Rationale:    {str(row["role_rationale"])[:150]}')
        print()

## 9. Top abstracted social roles — what role vocabulary emerged?

In [ ]:
# Extract the first few words of each subject to find common role patterns
def extract_role_prefix(subj, n_words=5):
    if pd.isna(subj):
        return ''
    words = subj.lower().split()[:n_words]
    return ' '.join(words)

processed['role_prefix'] = processed['raz_norm_subject'].apply(extract_role_prefix)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 20 role prefixes (all novels)
top_roles = processed['role_prefix'].value_counts().head(20)
top_roles.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_xlabel('Count')
axes[0].set_title('Top 20 Abstracted Role Prefixes (5-word)')
axes[0].invert_yaxis()

# Unique roles per novel
roles_per_book = processed.groupby('book_title')['raz_norm_subject'].nunique().sort_values()
roles_per_book.plot.barh(ax=axes[1], color='darkorange')
axes[1].set_xlabel('Unique subjects')
axes[1].set_title('Subject Diversity by Novel')

plt.tight_layout()
plt.show()

print(f'Total unique subjects: {processed["raz_norm_subject"].nunique():,}')
print(f'(vs. {processed["orig_raz_norm_subject"].nunique():,} before abstraction)')

## 10. Embedding readiness — full before/after articulation comparison

In [ ]:
processed['orig_art_len'] = processed['orig_raz_norm_articulation'].str.len()
processed['new_art_len'] = processed['raz_norm_articulation'].str.len()
processed['art_len_delta'] = processed['new_art_len'] - processed['orig_art_len']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length delta histogram
processed['art_len_delta'].hist(bins=80, ax=axes[0], edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Articulation length change (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Articulation Length Delta (new - original)')

# Per-novel mean delta
delta_by_book = processed.groupby('book_title')['art_len_delta'].mean().sort_values()
delta_by_book.plot.barh(ax=axes[1], color=['#d62728' if v < 0 else '#2ca02c' for v in delta_by_book])
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_xlabel('Mean articulation length change (chars)')
axes[1].set_title('Articulation Length Delta by Novel')

plt.tight_layout()
plt.show()

print(f'Articulation length: orig mean={processed["orig_art_len"].mean():.0f}, new mean={processed["new_art_len"].mean():.0f}')
print(f'Mean delta: {processed["art_len_delta"].mean():.1f} chars')

## 11. Summary

In [ ]:
n_total = len(processed)
n_changed = processed['any_changed'].sum()
n_subj_changed = processed['subject_changed'].sum()

# Character name removal stats (recompute on full set)
orig_has_names = processed['orig_raz_norm_articulation'].apply(has_character_name).sum()
new_has_names = processed['raz_norm_articulation'].apply(has_character_name).sum()

print('=' * 60)
print('ROLE ABSTRACTION SUMMARY')
print('=' * 60)
print(f'Total norms processed:          {n_total:,}')
print(f'Abstraction failures:           {df["role_abstraction_failed"].sum():,}')
print(f'Norms with any field rewritten: {n_changed:,} ({n_changed/n_total*100:.1f}%)')
print(f'Subjects rewritten:             {n_subj_changed:,} ({n_subj_changed/n_total*100:.1f}%)')
print()
print(f'Character names in articulation:')
print(f'  Before: {orig_has_names:,} ({orig_has_names/n_total*100:.1f}%)')
print(f'  After:  {new_has_names:,} ({new_has_names/n_total*100:.1f}%)')
print(f'  Reduction: {(1 - new_has_names/max(orig_has_names,1))*100:.1f}%')
print()
print(f'Subject length:')
print(f'  Before: mean={processed["orig_subj_len"].mean():.0f} chars')
print(f'  After:  mean={processed["new_subj_len"].mean():.0f} chars')
print()
print(f'Unique subjects:')
print(f'  Before: {processed["orig_raz_norm_subject"].nunique():,}')
print(f'  After:  {processed["raz_norm_subject"].nunique():,}')
print()
print('Per-novel breakdown:')
summary = processed.groupby('book_title').agg(
    norms=('raz_norm_index', 'count'),
    pct_rewritten=('any_changed', 'mean'),
    mean_subj_len=('new_subj_len', 'mean'),
    unique_subjects=('raz_norm_subject', 'nunique'),
).sort_values('norms', ascending=False)
summary['pct_rewritten'] = (summary['pct_rewritten'] * 100).round(1)
summary['mean_subj_len'] = summary['mean_subj_len'].round(0).astype(int)
print(summary.to_string())